# PerGAZE ChenLSTMISP on Google Colab: one GPU

Select **Runtime > Change runtime type > GPU**, then run cells in order.
Upload the PerGAZE dataset to Google Drive and edit DATA_SOURCE below.
The notebook always uses cuda:0, regardless of additional visible GPUs.

Train: train.json. Validation: test_seen.json. Default schedule: 5 supervised
and 5 RL epochs, with one warmup epoch. The notebook does not train/evaluate
support.json or test_unseen.json. TP/TA observers with the same numeric ID are
different people. Object branches select one of 18 COCO convolutions; VQA uses
the separate AIR convolution. Original images and attention maps are loaded
on demand; no precomputed image feature cache or text embeddings are needed.

The main train cell downloads ImageNet-pretrained ResNet50 weights if not cached.
The smoke check uses random weights solely to verify execution. Checkpoints,
config and epoch/evaluation reports are saved to RUN on Drive for resume.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import subprocess
import sys
from pathlib import Path

CODE = Path("/content/IndividualScanpath")
if (CODE / ".git").is_dir():
    subprocess.run(["git", "-C", str(CODE), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/ntnghia06/IndividualScanpath.git", str(CODE)], check=True)
SRC = CODE / "PerGAZE/ChenLSTMISP/src"
assert (SRC / "train.py").is_file()
assert (SRC / "dataset/tasks.py").is_file(), "Update the code for COCO object-head support"


In [ ]:
%pip install -q "numpy>=1.24" "Pillow>=9.1" "scipy>=1.10" "tqdm>=4.65" multimatch-gaze scikit-image


In [ ]:
import torch

assert torch.cuda.is_available(), "Select GPU in Runtime > Change runtime type"
GPU_IDS = [0]
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory GiB:", torch.cuda.get_device_properties(0).total_memory / 2**30)


## Configure dataset and run

Example Drive layout:

```text
MyDrive/PerGAZE/dataset/
  train.json
  test_seen.json
  images/
  attention_reasoning/
```

Edit DATA_SOURCE and RUN. COPY_DATA_TO_LOCAL=True stages original images,
attention maps and JSON files to /content to reduce small-file Drive reads.
Set False to read inputs directly from Drive. This does not extract features.
RUN stays on Drive; it saves last/best checkpoints after each completed epoch.

To resume after interruption, point RUN at the existing run and set RESUME=True.
Keep seed, train batch, max_length, RL schedule and input JSON contents consistent
with that run. A new runtime may use GPU 0 instead of the previous GPU topology.
Interrupted progress within an unsaved epoch restarts from the last completed
checkpoint. Use a new RUN for another experiment.


In [ ]:
SEED = 1
EPOCHS = 10
START_RL_EPOCH = 5  # Zero-based: epochs 1-5 supervised, 6-10 RL.
MAX_LENGTH = 16
TRAIN_BATCH = 2
TEST_BATCH = 1
WORKERS = 2
RESUME = False
COPY_DATA_TO_LOCAL = True
VALIDATE_DATA = True
RUN_SMOKE = True

DRIVE_ROOT = Path("/content/drive/MyDrive/PerGAZE")
DATA_SOURCE = DRIVE_ROOT / "dataset"
RUN = DRIVE_ROOT / "runs/pergaze_chenlstm_colab_1gpu"
LOCAL_ROOT = Path("/content/pergaze_chenlstm_inputs")

for name in ("train.json", "test_seen.json", "images", "attention_reasoning"):
    assert (DATA_SOURCE / name).exists(), f"Missing: {DATA_SOURCE / name}"
assert min(EPOCHS, MAX_LENGTH, TRAIN_BATCH, TEST_BATCH) >= 1
assert START_RL_EPOCH >= 0 and WORKERS >= 0
if RESUME:
    assert (RUN / "checkpoints/checkpoint.pth").is_file(), f"No resume checkpoint at {RUN}"


In [ ]:
import shutil

if COPY_DATA_TO_LOCAL:
    DATA = LOCAL_ROOT / "dataset"
    DATA.mkdir(parents=True, exist_ok=True)
    for name in ("train.json", "test_seen.json"):
        shutil.copy2(DATA_SOURCE / name, DATA / name)
    for name in ("images", "attention_reasoning"):
        print(f"Copying original {name} from Drive to /content...")
        shutil.copytree(DATA_SOURCE / name, DATA / name, dirs_exist_ok=True)
else:
    DATA = DATA_SOURCE
RUN.mkdir(parents=True, exist_ok=True)
print("Dataset:", DATA)
print("Persistent Drive output:", RUN)


In [ ]:
COMMON = [
    "--train_file", str(DATA / "train.json"),
    "--val_file", str(DATA / "test_seen.json"),
    "--img_dir", str(DATA / "images"),
    "--att_dir", str(DATA / "attention_reasoning"),
    "--seed", str(SEED), "--max_length", str(MAX_LENGTH),
    "--device", "cuda:0", "--gpu_ids", "0",
]

def run_script(script, *arguments):
    subprocess.run([sys.executable, "-u", str(SRC / script), *COMMON, *arguments], check=True)


## Validate data

Validate records, images, TP boxes and VQA attention maps before training.
This does not update model weights. VALIDATE_DATA=False skips this stage when
reusing already validated input files.


In [ ]:
if VALIDATE_DATA:
    run_script("preprocess/validate_dataset.py", "--log_root", str(RUN))


## Optional smoke check

RUN_SMOKE=True checks forward/backward/inference and RL on real samples from
TP/TA/VQA using random weights and length 3. It writes only a smoke report to
/content, not a trained checkpoint. This check is skipped when resuming.


In [ ]:
if RUN_SMOKE and not RESUME:
    run_script("train.py", "--smoke_test", "--batch", "1", "--max_length", "3",
               "--rl_sample_number", "2", "--log_root", "/content/pergaze_chenlstm_smoke")


## Train or resume

The main cell uses the configured batch and max_length and the pretrained
ImageNet ResNet50. Defaults preserve the 5 supervised + 5 RL schedule.
Validation runs each epoch, with best selected by the harmonic mean of the
ScanMatch means. Progress bars show training, validation and subject retrieval.
RESUME restores saved model/optimizer/scheduler/RNG; incompatible settings fail
explicitly instead of silently starting a different experiment.


In [ ]:
arguments = ["--log_root", str(RUN), "--workers", str(WORKERS),
             "--batch", str(TRAIN_BATCH), "--test_batch", str(TEST_BATCH),
             "--epoch", str(EPOCHS), "--start_rl_epoch", str(START_RL_EPOCH)]
if RESUME:
    arguments.append("--resume")
run_script("train.py", *arguments)


## Evaluate the best checkpoint

Evaluate best.pth on test_seen.json, which is the validation split. This is
not an independent unseen-user test. The checkpoint restores architecture and
subject mapping. Reports include ScanMatch, MultiMatch, SED/STDE and retrieval.


In [ ]:
run_script("test.py", "--log_root", str(RUN), "--workers", str(WORKERS),
           "--test_batch", str(TEST_BATCH))


In [ ]:
import json
report_path = RUN / "evaluation_validation.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
print(json.dumps(report["overall"], indent=2))
print(json.dumps(report.get("retrieval", {}).get("by_condition", {}), indent=2))
print("Full evaluation report:", report_path)


## Persistent outputs and optional download

RUN on Drive contains hparams.json, manifest.json, epoch_*.json,
checkpoints/checkpoint.pth (last), best.pth and supervised.pth, plus evaluation
and prediction JSON files. Preserve the whole run directory for resume.
The next cell prints its path; uncomment the remaining lines to create a ZIP
in /content and download it. No copy-back step is needed for resume: RUN already
points to persistent Drive storage.


In [ ]:
print("Outputs saved on Google Drive:", RUN)
# LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
# archive = shutil.make_archive(str(LOCAL_ROOT / "pergaze_chenlstm_results"),
#                              "zip", root_dir=RUN.parent, base_dir=RUN.name)
# from google.colab import files
# files.download(archive)
